In [0]:
df_online_bronze = spark.read.table("workspace.superstore.bronze_online_superstore")
df_online_bronze.createOrReplaceTempView("bronze_superstore_view")

df_online_silver = (spark.sql("""SELECT 
    `Row ID` AS row_id,
    `Order ID` AS receipt_id,
    to_timestamp(`Order Date`, 'M/d/yyyy') AS transaction_timestamp,
    to_timestamp(`Ship Date`, 'M/d/yyyy') AS ship_date,
    `Ship Mode` AS ship_mode,
    TRIM(`Customer ID`) AS customer_id,
    `Customer Name` AS customer_name,
    `Segment` AS segment,
    `Country` AS country,
    `City` AS city,
    `State` AS state,
    `Postal Code` AS postal_code,
    `Region` AS region,
    TRIM(`Product ID`) AS product_id,
    `Category` AS category,
    `Sub-Category` AS sub_category,
    `Product Name` AS product_name,
    `Sales` AS sales_price,
    `Quantity` AS quantity,
    `Discount` AS discount,
    `Profit` AS profit
    FROM bronze_superstore_view
    WHERE `Order ID` is not null
                          """))

df_online_silver = df_online_silver.dropDuplicates()


In [0]:
(df_online_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.superstore.silver_online_superstore") 
)

In [0]:
display(df_online_silver)